# RAPTOR on BookSum

End-to-end demo: load a BookSum chapter → build RAPTOR tree → generate summary or answer a question → evaluate against the reference summary.

**Why BookSum?**  
Chapters are long enough that a plain LLM call must truncate, making RAPTOR's hierarchical retrieval genuinely advantageous.

## Section 0 — Setup

In [28]:
import os, sys, importlib
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.path.abspath('.')), '..', '.env'))
load_dotenv()  # fallback

_root = os.path.abspath('..')
sys.path.insert(0, os.path.join(_root, 'lib'))        # for `from raptor import ...`
sys.path.insert(0, os.path.join(_root, 'pipelines'))  # for pipeline modules

print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

OPENAI_API_KEY set: True


## Section 1 — Load BookSum chapter

We filter to single chapters only (`is_aggregate=False`) so each row is one chapter with a matching reference summary.

In [29]:
import booksum_pipeline
importlib.reload(booksum_pipeline)
from booksum_pipeline import load_booksum, extract_booksum_text

dataset = load_booksum(split='validation')
print(f'Chapters available: {len(dataset)}')

row = dataset[0]
text, reference, title = extract_booksum_text(row)

print(f'Title     : {title}')
print(f'Words     : {len(text.split())}')
print(f'Ref words : {len(reference.split())}')
print(f'\nChapter preview:\n{text[:400]}...')

Chapters available: 1350
Title     : Bleak House.chapter 18
Words     : 7056
Ref words : 243

Chapter preview:
Lady Dedlock


It was not so easy as it had appeared at first to arrange for
Richard's making a trial of Mr. Kenge's office. Richard himself was
the chief impediment. As soon as he had it in his power to leave Mr.
Badger at any moment, he began to doubt whether he wanted to leave
him at all. He didn't know, he said, really. It wasn't a bad
profession; he couldn't assert that he disliked it; perhap...


## Section 2 — Configure RAPTOR

In [30]:
import qasper_pipeline
importlib.reload(qasper_pipeline)
from qasper_pipeline import build_raptor_config

config = build_raptor_config()
print('Config ready.')

2026-04-07 17:16:27,627 - Use pytorch device_name: cpu
2026-04-07 17:16:27,627 - Load pretrained SentenceTransformer: sentence-transformers/multi-qa-mpnet-base-cos-v1
2026-04-07 17:16:27,737 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-mpnet-base-cos-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:16:27,751 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-mpnet-base-cos-v1/d51b22a1dfa8184e9258074e56e2875e50612dca/modules.json "HTTP/1.1 200 OK"
2026-04-07 17:16:27,871 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/multi-qa-mpnet-base-cos-v1/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:16:27,892 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/multi-qa-mpnet-base-cos-v1/d51b22a1dfa8184e9258074e56e2875e50612dca/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-0

Config ready.


## Section 3 — Summarize chapter

In [31]:
importlib.reload(booksum_pipeline)
import qasper_pipeline
importlib.reload(qasper_pipeline)
from booksum_pipeline import build_tree
from prompts import DOCUMENT_SUMMARY_SYSTEM, DOCUMENT_SUMMARY_USER
from openai import OpenAI

# force_rebuild=True discards the old cache and rebuilds with the current config.
# Set back to False once you're happy with the config.
ra = build_tree(row, config=config, force_rebuild=True)

# Retrieve and summarise — result is available for Section 5
context, _ = ra.retrieve(
    question='Summarize this chapter',
    top_k=20,
    max_tokens=10000,
    collapse_tree=True,
    return_layer_information=True,
)

client = OpenAI()
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role': 'system', 'content': DOCUMENT_SUMMARY_SYSTEM},
        {'role': 'user', 'content': DOCUMENT_SUMMARY_USER.format(context=context)},
    ],
    max_tokens=500,
)
summary = response.choices[0].message.content.strip()
result = {'summary': summary}

print(f'Tree layers : {len(ra.tree.layer_to_nodes)}')
print(f'Total nodes : {sum(len(v) for v in ra.tree.layer_to_nodes.values())}')
print(f'\nSUMMARY:\n{summary}')

2026-04-07 17:16:30,133 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
            Selection Mode: threshold
            Summarization Length: 300
            Summarization Model: <raptor.SummarizationModels.GPT4NanoSummarizationModel object at 0x000001FE2C84B950>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.SBertEmbeddingModel object at 0x000001FEFCED5CD0>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-04-07 17:16:30,134 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
     

Building tree for: Bleak House.chapter 18


Batches:   0%|          | 0/1 [00:00<?, ?it/s]








































































































Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.65s/it]



Batches: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.75s/it]



Batches: 100%|██████████| 1/1 [00:02<00:00,  2.78s/it]


















Batches: 100%|██████████| 1/1 [00:02<00:00,  2.82s/it]








Batches: 100%|██████████| 1/1 [00:02<00:00,  2.84s/it]































Batches: 100%|██████████| 1/1 [00:02<00:00,  2.87s/it]























Batches: 100%|██████████| 1/1 [00:02<00:00,  2.90s/it]



















Batches: 100%|██████████| 1/1 [00:02<00:00,  3.00s/it]























Batches: 100%|██████████| 1/1 [00:03<00:00,  3.03s/it]
















































































Batches: 100%|██████████| 1/1 [00:02<00:

Tree saved to: c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\pipelines\..\.tree_cache\Bleak_House_chapter_18.pkl


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.54it/s]
2026-04-07 17:17:16,571 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Tree layers : 2
Total nodes : 54

SUMMARY:
The passage offers a multifaceted narrative involving characters from Dickens's novel, focusing on themes of social status, personal identity, and emotional undercurrents. Lady Dedlock displays a nuanced attitude toward a Pretty Girl and her Frenchmaid, demonstrating calmness and subtlety in her interactions, while maintaining her aloof grace. The scene depicts her observing the rainy landscape and engaging with visitors, including a Frenchwoman who claims to be her maid. Lady Dedlock dismisses the girl with composed authority before departing, hinting at past conflicts and complex relationships. 

The narrative also showcases eccentric characters like Mr. Boythorn, whose humorous exaggerations about defending his estate contrast with more reserved characters like Sir Leicester Dedlock, whose marriage is viewed as mysterious. Skimpole's carefree, superficial attitude toward social principles and his remarks on adventure and duty reflect a ligh

## Section 4 — Query the chapter

Use RAPTOR for open-ended questions: *"What does this chapter say about X?"*  
The collapsed tree retrieval finds the most relevant nodes regardless of where in the chapter they appear.

In [32]:
importlib.reload(booksum_pipeline)
from booksum_pipeline import query_chapter

question = "What does this chapter say about the relationship between the characters?"

# Reuses cached tree — no rebuild
answer = query_chapter(row, question=question, config=config)
print(f'Q: {question}')
print(f'\nA: {answer}')

2026-04-07 17:17:16,777 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
            Selection Mode: threshold
            Summarization Length: 300
            Summarization Model: <raptor.SummarizationModels.GPT4NanoSummarizationModel object at 0x000001FE2C84B950>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.SBertEmbeddingModel object at 0x000001FEFCED5CD0>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-04-07 17:17:16,778 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
     

Loading cached tree: c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\pipelines\..\.tree_cache\Bleak_House_chapter_18.pkl


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.24it/s]
2026-04-07 17:17:20,940 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Q: What does this chapter say about the relationship between the characters?

A: This chapter reveals a complex and nuanced relationship between the characters, marked by subtle distinctions of social standing, distant familiarity, and underlying emotional tension. Lady Dedlock appears to hold a superior and somewhat aloof attitude toward others, especially the young girl and Mr. Jarndyce, whom she recognizes with a mixture of politeness and detachment. Her interactions suggest an awareness of past associations and a reserved, self-possessed demeanor that hints at deeper personal secrets and unspoken connections.

The relationship between Lady Dedlock and the young girl (who is closely associated with Ada Clare) is somewhat paternal and protective on Lady Dedlock’s part, as she shows a particular fondness for this girl, intending to keep her close. However, her aloofness and the subtle withdrawal in her dismissing her from her presence indicate emotional distance and possibly unresolve

## Section 5 — Evaluate summary

In [33]:
importlib.reload(qasper_pipeline)
from qasper_pipeline import print_evaluation

scores = print_evaluation(result['summary'], reference)

2026-04-07 17:17:20,985 - Using default tokenizer.


REFERENCE:
In Chapter 18, Richard begins at Kenge and Carboy's. The rest of the party leave him in London, and go to visit Mr. Boythorn at his place down in Lincolnshire, which neighbors the Dedlocks' estate. When the party goes to church, Esther sees not only Rosa, Mrs. Rouncewell, and Mademoiselle Hortense, but also Sir Leicester and Lady Dedlock. Esther seems to recognize Lady Dedlock's face, and wonders at the resemblance to her godmother's face. She also vaguely realizes that Lady Dedlock's face resembles her own when seen in the mirror. Later, Esther, Ada, and John Jarndyce then go for a walk and take refuge in a gamekeeper's lodge when it rains. Lady Dedlock is already there, and Esther experiences another weird feeling upon meeting Lady Dedlock. Lady Dedlock and John Jarndyce are old acquaintances, but it appears Mr. Jarndyce was actually closer to Lady Dedlock's sister. They discuss Lady Dedlock's sister's death, as she "died in retirement. Esther admires Lady Dedlock's beauty

2026-04-07 17:17:21,249 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:21,451 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:21,578 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:21,690 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-07 17:17:21,802 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:21,922 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-07 17:17:22,576 - HTTP Re

SCORES:
  ROUGE-1  : 0.2581
  ROUGE-2  : 0.0419
  ROUGE-L  : 0.1252
  BERTScore: P=0.8220  R=0.8275  F1=0.8247


## Section 6 — Benchmark: RAPTOR vs Baseline

Compare RAPTOR against a plain LLM call (same model, same prompt, raw truncated text).

In [34]:
importlib.reload(booksum_pipeline)
from booksum_pipeline import compare_chapter

results = compare_chapter(row, config=config)

2026-04-07 17:17:25,047 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
            Selection Mode: threshold
            Summarization Length: 300
            Summarization Model: <raptor.SummarizationModels.GPT4NanoSummarizationModel object at 0x000001FE2C84B950>
            Embedding Models: {'EMB': <raptor.EmbeddingModels.SBertEmbeddingModel object at 0x000001FEFCED5CD0>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-04-07 17:17:25,048 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 200
            Num Layers: 3
            Threshold: 0.1
            Top K: 20
     

Loading cached tree: c:\Users\mickv\OneDrive\Bureaublad\Gradient\Summarization\Code\pipelines\..\.tree_cache\Bleak_House_chapter_18.pkl


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.60it/s]

Document: Bleak House.chapter 18


Running baseline...


2026-04-07 17:17:28,525 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-07 17:17:28,533 - Using default tokenizer.
2026-04-07 17:17:28,739 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:28,923 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:29,056 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:29,167 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-07 17:17:29,281 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:29,394 - HTTP Requ

Using pre-built RAPTOR context...


2026-04-07 17:17:35,711 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-07 17:17:35,723 - Using default tokenizer.
2026-04-07 17:17:36,024 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:36,151 - HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-07 17:17:36,296 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:36,638 - HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-07 17:17:36,752 - HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-07 17:17:36,861 - HTTP Requ


BASELINE SUMMARY (plain LLM, no RAPTOR):
--------------------------------------------------------------------------------
The document recounts a series of detailed episodes centered around characters from Charles Dickens' novel "Bleak House." It follows Richard, whose vacillations between law and medicine delay his career choice, and his humorous misadventures as he begins experiments in lawyering. Lady Dedlock makes a notable appearance, revealing her graceful yet aloof demeanor and a mysterious, emotional encounter with the narrator, Esther Summerson, who is haunted by her past and a resemblance to her godmother. The narrative describes a journey through picturesque rural England, visiting friends like Mr. Boythorn and Ada, with vivid descriptions of their countryside estate, local church, and social interrelations. Intertwined are dialogues with Mr. Skimpole, a charming but irresponsible figure, discussing social harmony and his carefree attitude toward money and principle. The st